In [1]:
from openai import OpenAI
import os

# 初始化OpenAI客户端
client = OpenAI(
    # 若没有配置环境变量，请用阿里云百炼API Key将下行替换为：api_key="sk-xxx",
    # 各地域的API Key不同。获取API Key：https://help.aliyun.com/zh/model-studio/get-api-key
    api_key="sk-bfc69abc05ae4a91bd6e172fddaa9a35",
    # 以下是北京地域base_url，若使用弗吉尼亚地域模型，需要将base_url换成https://dashscope-us.aliyuncs.com/compatible-mode/v1
    # 如果使用新加坡地域的模型，需要将base_url替换为：https://dashscope-intl.aliyuncs.com/compatible-mode/v1
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

In [2]:
reasoning_content = ""  # 定义完整思考过程
answer_content = ""     # 定义完整回复
is_answering = False   # 判断是否结束思考过程并开始回复
enable_thinking = True
# 创建聊天完成请求
completion = client.chat.completions.create(
    model="qwen3.5-plus",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://img.alicdn.com/imgextra/i1/O1CN01gDEY8M1W114Hi3XcN_!!6000000002727-0-tps-1024-406.jpg"
                    },
                },
                {"type": "text", "text": "这道题怎么解答？"},
            ],
        },
    ],
    stream=True,
    # enable_thinking 参数开启思考过程，thinking_budget 参数设置最大推理过程 Token 数
    # qwen3.5-plus、qwen3-vl-plus、qwen3-vl-flash可通过enable_thinking开启或关闭思考（其中qwen3.5-plus默认开启）、对于qwen3-vl-235b-a22b-thinking等带thinking后缀的模型，enable_thinking仅支持设置为开启，对其他Qwen-VL模型均不适用
    extra_body={
        'enable_thinking': enable_thinking},

    # 解除以下注释会在最后一个chunk返回Token使用量
    # stream_options={
    #     "include_usage": True
    # }
)

if enable_thinking:
    print("\n" + "=" * 20 + "思考过程" + "=" * 20 + "\n")

for chunk in completion:
    # 如果chunk.choices为空，则打印usage
    if not chunk.choices:
        print("\nUsage:")
        print(chunk.usage)
    else:
        delta = chunk.choices[0].delta
        # 打印思考过程
        if hasattr(delta, 'reasoning_content') and delta.reasoning_content != None:
            print(delta.reasoning_content, end='', flush=True)
            reasoning_content += delta.reasoning_content
        else:
            # 开始回复
            if delta.content != "" and is_answering is False:
                print("\n" + "=" * 20 + "完整回复" + "=" * 20 + "\n")
                is_answering = True
            # 打印回复过程
            print(delta.content, end='', flush=True)
            answer_content += delta.content

# print("=" * 20 + "完整思考过程" + "=" * 20 + "\n")
# print(reasoning_content)
# print("=" * 20 + "完整回复" + "=" * 20 + "\n")
# print(answer_content)


====================思考过程====================

这个用户想要解决图片中的数学题。

**1. 分析题目要求：**
*   **标题：** 五、分别计算下列图形的表面积和体积。（单位：cm）（每题 6 分，共 12 分）
*   **任务：** 需要计算两个几何体的表面积和体积。
*   **单位：** 厘米 (cm)。

**2. 分析第一个图形 (图1)：**
*   **形状：** 长方体 (Rectangular Prism / Cuboid)。
*   **尺寸：**
    *   长 (Length, $l$) = 4 cm
    *   宽 (Width, $w$) = 3 cm
    *   高 (Height, $h$) = 2 cm
*   **公式回顾：**
    *   长方体表面积公式：$S = 2 \times (lw + lh + wh)$
    *   长方体体积公式：$V = l \times w \times h$
*   **计算过程：**
    *   表面积：
        *   $lw = 4 \times 3 = 12$
        *   $lh = 4 \times 2 = 8$
        *   $wh = 3 \times 2 = 6$
        *   和 = $12 + 8 + 6 = 26$
        *   总表面积 = $26 \times 2 = 52$ cm²
    *   体积：
        *   $V = 4 \times 3 \times 2 = 24$ cm³

**3. 分析第二个图形 (图2)：**
*   **形状：** 正方体 (Cube)。虽然看起来像长方体，但标注的长、宽、高都是3。
*   **尺寸：**
    *   棱长 (Side length, $a$) = 3 cm
*   **公式回顾：**
    *   正方体表面积公式：$S = 6 \times a^2$
    *   正方体体积公式：$V = a^3$
*   **计算过程：**
    *   表面积：
        *   $a^2 = 3 \times 3 = 9$
        *   总表面积 = $

# 读本地图片

In [10]:
reasoning_content = ""  # 定义完整思考过程
answer_content = ""     # 定义完整回复
is_answering = False   # 判断是否结束思考过程并开始回复
enable_thinking = True

import base64

# 1. 定义一个函数：将本地图片编码为 Base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# 2. 你的本地图片路径
local_image_path = "/home/jiahaoli/homework/Week10/yrafcqtnvuolpkew.png"

# 3. 获取 Base64 编码
base64_image = encode_image(local_image_path)

# 4. 在 messages 中使用 data URL 格式
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image_url",
                "image_url": {
                    # 关键修改：使用 data URL 格式，而不是直接写路径
                    "url": f"data:image/png;base64,{base64_image}"
                },
            },
            {"type": "text", "text": "图中是什么？"},
        ],
    },
]


# 创建聊天完成请求
completion = client.chat.completions.create(
    model="qwen3.5-plus",
    messages=messages,
    stream=True,
    # enable_thinking 参数开启思考过程，thinking_budget 参数设置最大推理过程 Token 数
    # qwen3.5-plus、qwen3-vl-plus、qwen3-vl-flash可通过enable_thinking开启或关闭思考（其中qwen3.5-plus默认开启）、对于qwen3-vl-235b-a22b-thinking等带thinking后缀的模型，enable_thinking仅支持设置为开启，对其他Qwen-VL模型均不适用
    extra_body={
        'enable_thinking': enable_thinking},

    # 解除以下注释会在最后一个chunk返回Token使用量
    # stream_options={
    #     "include_usage": True
    # }
)

if enable_thinking:
    print("\n" + "=" * 20 + "思考过程" + "=" * 20 + "\n")

for chunk in completion:
    # 如果chunk.choices为空，则打印usage
    if not chunk.choices:
        print("\nUsage:")
        print(chunk.usage)
    else:
        delta = chunk.choices[0].delta
        # 打印思考过程
        if hasattr(delta, 'reasoning_content') and delta.reasoning_content != None:
            print(delta.reasoning_content, end='', flush=True)
            reasoning_content += delta.reasoning_content
        else:
            # 开始回复
            if delta.content != "" and is_answering is False:
                print("\n" + "=" * 20 + "完整回复" + "=" * 20 + "\n")
                is_answering = True
            # 打印回复过程
            print(delta.content, end='', flush=True)
            answer_content += delta.content

# print("=" * 20 + "完整思考过程" + "=" * 20 + "\n")
# print(reasoning_content)
# print("=" * 20 + "完整回复" + "=" * 20 + "\n")
# print(answer_content)


====================思考过程====================

用户问的是“图中是什么？”，我需要仔细观察图片并提供详细的描述。

1.  **整体类型**：这是一张折线图（line chart）。
2.  **标题**：图表顶部有一个标题 "sr:2.062669 sr_is:1.873694"。这看起来像是某种统计指标或者模型参数的缩写，可能是夏普比率（Sharpe Ratio, sr）和某种信息比率（Information Ratio, sr_is? 或者可能是排序相关的指标）。
3.  **X轴（横轴）**：
    *   标签是 "date"（日期）。
    *   时间跨度从 2015-01-05 到 2025-04-18。
    *   刻度包括：2015-01-05, 2017-01-19, 2019-02-13, 2021-03-04, 2023-03-24, 2025-04-18。这表明这是一个长达10年的时间序列数据。
4.  **Y轴（纵轴）**：
    *   没有明确的标签，但数值从 0 到 4000。
    *   刻度包括：0, 1000, 2000, 3000, 4000。
    *   这通常代表价格、指数点位、或者累积收益。
5.  **数据趋势**：
    *   蓝色的线条显示了一个明显的上升趋势。
    *   从2015年初接近0的位置开始。
    *   在2015-2016年有一个快速上涨，然后横盘震荡。
    *   从2019年左右开始加速上涨。
    *   一直持续到2025年，达到了4000以上。
    *   整体来看，这是一个典型的长期牛市或者资产增值的走势图。
6.  **推测内容**：
    *   考虑到时间跨度（2015-2025）和走势，这很像是一个股票指数（如标普500、纳斯达克，或者中国的沪深300等，但沪深300在2015年很高，这个图是从0开始的，所以更像是累积收益率曲线或者某个特定策略的净值曲线）。
    *   标题中的 "sr" 通常指 Sharpe Ratio（夏普比率），这是金融量化领域常用的指标。这强烈暗示这是一张**量化交易策略的回测结果图**或者**基金净值走势图**。
    *   起始点接近0，然后迅速拉升

KeyboardInterrupt: 

# 理解发票pdf

In [14]:
from pdf2image import convert_from_path

# ========== 1. PDF 转图片（仅第一页）==========
def pdf_first_page_to_base64(pdf_path, dpi=200):
    """
    将 PDF 的第一页转换为 Base64 编码的 Data URL
    """
    # first_page=1 表示只转换第1页（pdf2image 索引从1开始）
    images = convert_from_path(pdf_path, dpi=dpi, first_page=1, last_page=1)
    
    if not images:
        raise ValueError("PDF 没有第一页")
    
    # 将 PIL Image 转换为 Base64
    import io
    buffer = io.BytesIO()
    images[0].save(buffer, format="JPEG")
    base64_image = base64.b64encode(buffer.getvalue()).decode('utf-8')
    
    return f"data:image/jpeg;base64,{base64_image}"

# ========== 2. 使用示例 ==========
pdf_path = "/home/jiahaoli/homework/Week10/dzfp_26352000000487241776_上海玉数投资管理有限公司_20260228104944.pdf"

# 获取第一页的 Data URL
data_url = pdf_first_page_to_base64(pdf_path, dpi=200)

In [15]:
completion = client.chat.completions.create(
    model="qwen-vl-plus",  # 或 qvq-max, qwen2.5-vl-72b-instruct
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": data_url}
                },
                {"type": "text", "text": "请问这个发票中的开票方是什么省份，发票金额多少"},
            ],
        },
    ],
)

print(completion.choices[0].message.content)

根据发票上的信息，我们可以得出以下结论：

1. **开票方省份**：
   - 发票上的销售方信息显示，销售方名称为“霞浦北礐欣海阁民宿有限公司”。
   - 从发票的红色印章中可以看到“福建省税务局”的字样。
   - 因此，开票方所在的省份是**福建省**。

2. **发票金额**：
   - 发票中的金额部分显示：
     - 项目名称为“住宿服务*住宿费”，金额为 **¥224.75**。
     - 税额为 **¥2.25**。
     - 合计金额为 **¥227.00**（大写为“贰佰贰拾柒圆整”，小写为“¥227.00”）。

### 总结：
- **开票方省份**：福建省
- **发票金额**：¥227.00
